In [760]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split 
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer                
from sklearn.linear_model import LogisticRegression

In [761]:
data = 'https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv'
!wget $data

--2025-10-10 09:12:04--  https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 80876 (79K) [text/plain]
Saving to: ‘course_lead_scoring.csv.22’

course_lead_scoring 100%[===================>]  78.98K  --.-KB/s    in 0.02s   

2025-10-10 09:12:04 (4.98 MB/s) - ‘course_lead_scoring.csv.22’ saved [80876/80876]



In [762]:
df = pd.read_csv(data)
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [763]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)
for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

In [764]:
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [765]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               1334 non-null   object 
 1   industry                  1328 non-null   object 
 2   number_of_courses_viewed  1462 non-null   int64  
 3   annual_income             1281 non-null   float64
 4   employment_status         1362 non-null   object 
 5   location                  1399 non-null   object 
 6   interaction_count         1462 non-null   int64  
 7   lead_score                1462 non-null   float64
 8   converted                 1462 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 102.9+ KB


In [766]:
df_filled = df.copy()
for col in categorical:
    df_filled[col] = df_filled[col].fillna('NA')
for col in numerical:
    df_filled[col] = df_filled[col].fillna(0.0)


In [767]:
df.isnull().sum()

lead_source                 128
industry                    134
number_of_courses_viewed      0
annual_income               181
employment_status           100
location                     63
interaction_count             0
lead_score                    0
converted                     0
dtype: int64

In [768]:
df.industry.value_counts()

industry
retail           203
finance          200
other            198
healthcare       187
education        187
technology       179
manufacturing    174
Name: count, dtype: int64

In [769]:
numerical = ['number_of_courses_viewed', 'interaction_count', 'lead_score']
categorical = ['lead_source', 'industry', 'employment_status', 'location']
features = numerical + categorical

In [770]:
df.head(9)

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1
5,events,manufacturing,1,59904.0,NaN,africa,6,0.83,1
6,social_media,technology,0,51283.0,NaN,middle_east,2,0.57,0
7,social_media,NaN,5,62975.0,student,europe,4,0.62,1
8,referral,healthcare,4,38648.0,unemployed,south_america,2,0.86,1


In [771]:
df[numerical].corrwith(df.converted)

number_of_courses_viewed    0.435914
interaction_count           0.374573
lead_score                  0.193673
dtype: float64

In [772]:
correlation_matrix = df[numerical].corr()
correlation_matrix

,number_of_courses_viewed,interaction_count,lead_score
number_of_courses_viewed,1.000000,-0.023565,-0.004879
interaction_count,-0.023565,1.000000,0.009888
lead_score,-0.004879,0.009888,1.000000


In [773]:
df_temp, df_train, y_temp, y_train = train_test_split(
    df_filled.drop('converted', axis=1), 
    df_filled['converted'], 
    test_size=0.6,  
    random_state=42, 
    stratify=df_filled['converted']
)

In [774]:
df_val.isnull().sum()

lead_source                  0
industry                     0
number_of_courses_viewed     0
annual_income               37
employment_status            0
location                     0
interaction_count            0
lead_score                   0
dtype: int64

In [775]:
df_val, df_test, y_val, y_test = train_test_split(
    df_temp, y_temp,
    test_size=0.5,  
    random_state=42,
    stratify=y_temp
)
df_test

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score
1363,organic_search,retail,2,67187.0,self_employed,middle_east,4,0.37
663,events,manufacturing,4,75559.0,self_employed,south_america,2,0.72
427,referral,healthcare,0,70164.0,employed,asia,4,0.35
1027,events,healthcare,2,48075.0,unemployed,asia,4,0.74
795,social_media,NA,3,49364.0,student,north_america,2,0.58
...,...,...,...,...,...,...,...,...
981,social_media,healthcare,1,64084.0,unemployed,middle_east,0,0.41
696,referral,retail,4,NaN,student,australia,2,0.49
447,events,education,0,49003.0,self_employed,south_america,5,0.32
474,referral,healthcare,0,48902.0,student,south_america,4,0.36


In [776]:
mutual_info_score(df_full_train.converted, df_full_train.interaction_count)

0.0871285827253218

In [777]:
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [778]:
score = df_full_train[categorical].apply(mutual_info_converted_score)
score.sort_values(ascending=False)
round(score,2)

lead_source          0.02
industry             0.01
employment_status    0.01
location             0.00
dtype: float64

In [779]:
dv = DictVectorizer(sparse=False)

dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
X_val = dv.transform(df_val[features].to_dict(orient='records')) 

In [780]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [781]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train,y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [782]:
model.intercept_[0]

-3.641129230502741

In [783]:
model.coef_[0].round(2)

array([-1.  ,  0.21, -0.4 , -0.73, -1.72, -0.68,  0.19, -0.19, -0.93,
       -0.12, -0.57, -0.62, -0.73,  1.02,  2.47, -0.63, -0.41, -0.67,
       -1.74,  0.61, -0.8 , -0.26, -0.97, -0.67, -0.25, -0.09, -0.61,
       -0.62, -0.18,  1.34])

In [784]:
y_pred = model.predict_proba(X_val)[:,1]
y_pred

array([1.69732326e-01, 3.81331915e-01, 8.42720953e-01, 4.86434109e-01,
       8.36838188e-01, 8.64270261e-01, 9.51090757e-01, 9.62513847e-01,
       4.83765660e-01, 1.37271858e-01, 9.25138302e-01, 6.08772997e-01,
       2.52915899e-01, 7.10950180e-01, 9.95528958e-01, 9.96799486e-01,
       8.47033896e-01, 7.88374103e-01, 9.94855587e-01, 6.27697998e-01,
       6.99231718e-01, 2.35670484e-02, 4.97339541e-02, 3.27494331e-01,
       9.64638322e-03, 9.98042764e-01, 3.21783926e-01, 9.99899718e-01,
       9.70305736e-01, 3.60067085e-01, 6.31991893e-01, 8.44175307e-01,
       8.57593900e-01, 4.82868320e-01, 2.09932695e-01, 6.97411932e-01,
       9.99060107e-01, 9.72652869e-01, 7.80623533e-01, 8.69618565e-03,
       7.79447822e-02, 9.60819897e-01, 7.84869625e-01, 6.53396648e-01,
       9.74498292e-01, 9.93647392e-01, 8.72906930e-01, 9.02778633e-01,
       9.65636677e-01, 4.68476308e-01, 3.81422655e-01, 3.79438497e-01,
       9.71238779e-01, 1.19204293e-01, 2.10362008e-01, 9.51172218e-01,
      

In [785]:
y_pred_val = model.predict(X_val)

accuracy = (y_pred_val == y_val).mean()
round(accuracy, 2) 

0.86

In [786]:
accuracy

0.8561643835616438

In [787]:
baseline_accuracy = accuracy

In [788]:
accuracy

0.8561643835616438

In [789]:
features
feature_importance = {}

In [790]:
features_to_check = ['industry', 'employment_status', 'lead_score']
drops = {f: feature_importance[f] for f in features_to_check}
least_useful = min(drops, key=drops.get)
print("Answer:", least_useful)

KeyError: 'industry'

In [ ]:
dicts_full_train = df_full_train[categorical + numerical].to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_full_train = dv.fit_transform(dicts_full_train)
y_full_train = df_full_train['converted'].values


dicts_test = df_test[categorical + numerical].to_dict(orient='records')
X_test = dv.transform(dicts_test)


best_acc = 0
best_C = None
for C in [0.01, 0.1, 1, 10, 100]:
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    acc = (model.predict(X_val) == y_val).mean()
    if acc > best_acc:
        best_acc = acc
        best_C = C
print(f"Best C: {best_C}, Accuracy: {best_acc:.3f}")